# Graph problems for quantum optimization

Build 14 classical graph problems using the same `QUBO` interface, inspect their
encodings, optimize a weighted independent set, decode the original graph
solution, and display the trained circuit with Matplotlib and TikZ.

Install or upgrade with `python -m pip install --upgrade "qasp[preview]"`.
Select that environment as your notebook kernel. The
[formulation guide](../../docs/graph_problems.md) gives every equation and penalty bound.

There are three distinct values: the training objective, penalized binary QUBO
energy, and original graph objective. A low surrogate loss does not certify a
valid graph solution. We will inspect each explicitly.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from qasp import Normal, QAOA, PCE, QUBO, draw_qaoa

torch.set_default_dtype(torch.float64)
assert hasattr(QUBO, "maximum_independent_set"), "Install the graph-formulations source update."
print(QUBO.formulations())

## 1. Choose a graph formulation

Vertices are numbered `0..N-1`. Pairs specify topology; supported weighted edge
problems also take `(u, v, weight)`. Vertex weights use a separate `weights=`
argument. Default penalties exceed a sufficient objective bound. New weighted
factories require finite nonnegative weights; MaxCut also permits signed weights.

These examples use small graphs so exact enumeration is quick, even for one-hot
and slack encodings. Change the problem, not the solver interface.

In [ ]:
def graph_problem_catalog():
    """Return one small instance of each named formulation, without training."""
    path = [(0, 1), (1, 2)]
    triangle = [*path, (0, 2)]
    costs = [(0, 1, 2.0), (1, 2, 3.0), (0, 2, 1.0)]
    return {
        "maxcut": QUBO.maxcut(3, costs),
        "maximum_independent_set": QUBO.maximum_independent_set(3, path, weights=[1, 1.5, 1]),
        "minimum_vertex_cover": QUBO.minimum_vertex_cover(3, path),
        "maximum_clique": QUBO.maximum_clique(3, triangle),
        "maximum_matching": QUBO.maximum_matching(3, costs),
        "minimum_dominating_set": QUBO.minimum_dominating_set(3, path),
        "minimum_edge_cover": QUBO.minimum_edge_cover(3, costs),
        "graph_coloring": QUBO.graph_coloring(3, triangle, n_colors=3),
        "max_k_cut": QUBO.max_k_cut(3, costs, n_parts=2),
        "graph_partition": QUBO.graph_partition(3, costs, subset_size=1),
        "densest_k_subgraph": QUBO.densest_k_subgraph(3, costs, k=2),
        "hamiltonian_path": QUBO.hamiltonian_path(3, path),
        "hamiltonian_cycle": QUBO.hamiltonian_cycle(3, triangle),
        "traveling_salesperson": QUBO.traveling_salesperson(3, costs),
    }

problems = graph_problem_catalog()

In [ ]:
rows = []
for name, problem in problems.items():
    bits, energy = problem.brute_force()
    decoded = problem.decode(bits)
    assert decoded.feasible and decoded.encoding_feasible
    rows.append({"problem": name, "vertices": problem.metadata["n_vertices"],
                 "decision bits": problem.n_decision_variables,
                 "total bits / QAOA qubits": problem.n_variables,
                 "direction": decoded.sense, "exact graph objective": decoded.objective,
                 "exact QUBO energy": energy})
display(pd.DataFrame(rows))

## 2. Check the binary layout and penalties

QAOA needs one qubit per **binary variable**, including auxiliary variables.
Coloring uses `N*K` variables and routes use `N*N`; the dense QUBO matrix then
uses the square of that binary count in memory. Exact enumeration and statevector
simulation become expensive quickly. Inspect `n_variables` before scaling up.

Dominating set uses coverage slacks. Both assignments below select a valid
vertex set, but only one encodes its slacks correctly. A decoder never silently
repairs the candidate. `penalty_is_sufficient=False` for a manually chosen
small penalty means the default bound no longer certifies it.

In [ ]:
cover = QUBO.minimum_dominating_set(2, [(0, 1)])
print("Layout:", cover.variable_labels)
print("Penalty:", cover.metadata["penalty"],
      "sufficient bound:", cover.metadata["sufficient_penalty_bound"])
for bits in ["1100", "1111"]:
    candidate = cover.decode(bits)
    print(bits, "graph feasible:", candidate.feasible,
          "encoding feasible:", candidate.encoding_feasible,
          "vertex cost:", candidate.objective, "QUBO energy:", candidate.energy)
    print(candidate.encoding_violations)

## 3. Configure a weighted independent-set experiment

The objective is to maximize selected vertex weight without selecting adjacent
vertices. Defaults choose a penalty greater than the largest weight. Initializer
`Normal(0, 0.2)` controls the cost and mixer angles, as elsewhere in QASP.
This problem needs no input encoding or input-scaling parameters.

In [ ]:
problem_config = {"n_vertices": 4, "edges": [(0, 1), (1, 2), (2, 3)],
                  "weights": [1, 2, 1, 2]}
model_config = {"n_layers": 2, "backend": "torch", "device": "default.qubit",
                "diff_method": "backprop", "init": Normal(0, 0.2), "seed": 7}
training_config = {"optimizer": "Adam", "optimizer_options": {"max_steps": 40, "lr": 0.05}}

problem = QUBO.maximum_independent_set(**problem_config)
model = QAOA(problem, **model_config)
start = model.initial_parameters()
print("Qubits:", model.n_qubits, "trainable angles:", model.n_parameters)
print("Initial gradient:", model.gradient(start))

## 4. Train and inspect the result

The same `QUBO` can be passed to standard, multi-angle, warm-start, linear-ramp
or recursive QAOA, or to PCE. Device compatibility depends on the circuit and
readout; a general QAOA circuit is not a commuting IQP circuit.

In [ ]:
result = model.solve(initial_parameters=start, **training_config)
candidate = problem.decode(result.bitstring)
exact_bits, exact_energy = problem.brute_force()  # small instance only
exact = problem.decode(exact_bits)
print("Selected vertices:", candidate.solution["vertices"])
print("Graph feasible:", candidate.feasible, "encoding feasible:", candidate.encoding_feasible)
print("Graph weight:", candidate.objective, "exact maximum:", exact.objective)
print("Penalized QUBO energy:", result.energy, "exact minimum:", exact_energy)
print("Expected energy during training:", result.objective)
print("Violations:", candidate.violations)
assert np.isfinite(result.parameters).all()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(np.arange(1, len(result.history) + 1), result.history, color="#1484c3")
axes[0].axhline(exact_energy, linestyle="--", color="gray", label="Exact binary minimum")
axes[0].set(xlabel="Optimizer step", ylabel="Expected penalized energy")
axes[0].legend()

n = problem_config["n_vertices"]
positions = np.column_stack((np.cos(np.arange(n)*2*np.pi/n), np.sin(np.arange(n)*2*np.pi/n)))
for u, v in problem_config["edges"]:
    axes[1].plot(*positions[[u, v]].T, color="#8796a8", zorder=1)
selected = set(candidate.solution["vertices"])
axes[1].scatter(*positions.T, s=850,
                c=["#1484c3" if v in selected else "#e4edf5" for v in range(n)], zorder=2)
for v, (x, y) in enumerate(positions):
    axes[1].text(x, y, f"{v}\nw={problem_config['weights'][v]}", ha="center", va="center",
                 color="white" if v in selected else "#22334a")
axes[1].set_title(f"Candidate vertex set · feasible={candidate.feasible}")
axes[1].set_aspect("equal")
axes[1].set_xlim(-1.4, 1.4)
axes[1].set_ylim(-1.4, 1.4)
axes[1].axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Display the trained model

Matplotlib is the default. The TikZ renderer displays a high-quality inline
preview using temporary files. `return_tex=False` avoids returning the source;
no figure is permanently saved. `standalone=True` supplies the document wrapper,
`symbolic_parameters=True` labels shared trainable angles, and `row_sep` controls
wire spacing. Install TeX with Quantikz and Ghostscript for the optional preview.
Use `save_tikz` separately when you want an exported file.

In [ ]:
fig, ax = draw_qaoa(model, parameters=result, measurement="probabilities")
display(fig)
plt.close(fig)

In [ ]:
draw_qaoa(model, parameters=result, measurement="probabilities", output="tikz",
          return_tex=False,
          tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"})

## 6. Reuse a formulation with PCE

PCE encodes the total QUBO variable count, including any slack bits, in Pauli
correlations. Its loss is a continuous surrogate: always check the original graph
constraints after decoding. General-QUBO fields use the documented extension of
the MaxCut loss and regularization scale. This small example demonstrates the API,
not a performance or quality benchmark between solvers.

In [ ]:
coloring = QUBO.graph_coloring(3, [(0, 1), (1, 2)], n_colors=2)
pce = PCE(coloring, backend="torch", device="default.qubit", diff_method="backprop",
          init=Normal(0, 0.2), seed=7)
pce_result = pce.solve(optimizer="Adam", optimizer_options={"max_steps": 12, "lr": 0.03})
color_candidate = coloring.decode(pce_result.bitstring)
print("QUBO bits:", coloring.n_variables, "PCE qubits:", pce.n_qubits)
print("Labels:", color_candidate.solution["colors"])
print("Feasible:", color_candidate.feasible, "violations:", color_candidate.violations)
print("Surrogate:", pce_result.objective, "QUBO energy:", color_candidate.energy)

## 7. Routes and directed graphs

For routing, bit index `v*N+t` places vertex v at position t. Both row and column
sums must be one. Cycles include the final return edge. Missing arcs are forbidden,
not free travel. The example is deliberately small: TSP uses N² binary variables.

In [ ]:
tsp = QUBO.traveling_salesperson(
    3, [(0, 1, 1), (1, 2, 2), (2, 0, 3), (1, 0, 4), (2, 1, 5), (0, 2, 6)],
    directed=True,
)
tour = tsp.decode(tsp.brute_force()[0])
print("Layout:", tsp.variable_labels)
print("Tour:", tour.solution["tour"], "cost:", tour.objective, "feasible:", tour.feasible)

## Next experiments

Change graph size and density, vertex/edge weights, depth, initializer or penalty,
then report original objective **and feasibility**, simulator choice, qubit count
and optimization budget. Use exact references only while the binary encoding is
small enough. A failed coloring or routing search is not proof the instance is
infeasible; our exhaustive unit tests establish that only for their small cases.

Standard reductions are described by [Lucas (2014)](https://doi.org/10.3389/fphy.2014.00005)
and [Glover, Kochenberger & Du (2018)](https://arxiv.org/abs/1811.11538).
The [QASP formulation guide](../../docs/graph_problems.md) specifies the equations,
penalty bounds, decoder semantics and resource requirements used here.